# First-24-hour intervention features

Create only `mechanical_ventilation_present`, `vasopressor_present` and `rrt_present` for the existing 45,253 ICU stays. All existing cohort, lab, vital and output artifacts are read-only. The output is `data/processed/intervention_features_first24h.parquet`.

## Sources and operational definitions

Definitions adapted from MIT-LCP's **MIMIC-III** concepts, reviewed 2026-09-11:
- [Ventilation classification](https://github.com/MIT-LCP/mimic-code/blob/main/mimic-iii/concepts/durations/ventilation_classification.sql): positive MechVent branches, supplemented by local D_ITEMS-confirmed procedure 225792 (Invasive Ventilation). Oxygen alone, extubation alone and procedure 225794 (non-invasive ventilation) are not positive evidence. This is the repository's ventilator-setting proxy, not an independently adjudicated invasive-ventilation label.
- [Vasopressor durations](https://github.com/MIT-LCP/mimic-code/blob/main/mimic-iii/concepts/durations/vasopressor_durations.sql): norepinephrine, epinephrine, phenylephrine, vasopressin and dopamine. Dobutamine and milrinone are excluded because this feature is vasopressor presence, not all vasoactive/inotropic therapy. No dose threshold beyond positive administration is applied to dopamine.
- [RRT pivot](https://github.com/MIT-LCP/mimic-code/blob/main/mimic-iii/concepts/pivot/pivoted_rrt.sql), [CRRT settings](https://github.com/MIT-LCP/mimic-code/blob/main/mimic-iii/concepts/durations/crrt_durations.sql), and [RRT sources](https://github.com/MIT-LCP/mimic-code/blob/main/mimic-iii/concepts/rrt.sql): treatment procedures, active machine/flow documentation, dialysis-specific administered fluids/medications and dialysis output. Access-line placement, dressing care, catheter history and diagnosis codes alone are excluded. Thus this is an explicitly stricter adaptation, not an exact reproduction of the broad historical RRT flag. RRT includes intermittent dialysis, CRRT/SCUF and peritoneal dialysis.

## Time and leakage policy

For every source, require **INTIME <= CHARTTIME (or STARTTIME) < INTIME + 24 hours**. Also require nonmissing **STORETIME < INTIME + 24 hours**. Records entered later or with unknown entry time are excluded. For interval input rows, positive rate is direct evidence; amount-only evidence is accepted only when ENDTIME is also within the observation window, so a future cumulative amount cannot trigger a feature. Procedure end times, durations and later extubation are not used. Pre-ICU start records are not carried forward; ongoing therapy must have qualifying in-window evidence. This conservative policy may undercount ongoing or late-documented therapy.

Exclude error-marked rows and canceled/rewritten MetaVision records. These are retrospective corrected tables, not an immutable real-time audit trail. `0` means no qualifying documented evidence; it is not proof that treatment never occurred. All three features are formed by OR across qualifying source records at ICUSTAY_ID level, with nonmissing SUBJECT_ID/HADM_ID checked against the unchanged cohort. Missing auxiliary IDs are allowed when ICUSTAY_ID matches; conflicting nonmissing IDs are excluded and counted.


In [1]:
import pandas as pd
from pathlib import Path
import hashlib
import time
from collections import Counter
import pyarrow as pa
import pyarrow.csv as pacsv
import pyarrow.compute as pc

# Existing data location, preserved from the initial notebook.
data_path = Path('/mnt/c/Users/vetts/Downloads/MimicIII/mimic-iii-clinical-database-1.4')
project_root = Path.cwd().resolve()
if project_root.name == 'notebooks':
    project_root = project_root.parent
processed = project_root / 'data/processed'
output_path = processed / 'intervention_features_first24h.parquet'

def digest(path):
    with path.open('rb') as handle:
        return hashlib.file_digest(handle, 'sha256').hexdigest()

protected_paths = sorted(p for p in processed.iterdir() if p.is_file() and p != output_path and not p.name.startswith('intervention_features_first24h.parquet.partial'))
protected_paths += sorted(p for p in (project_root/'notebooks').iterdir() if p.is_file() and p.name != '04_interventions.ipynb')
protected_before = {str(p): digest(p) for p in protected_paths}
cohort_path = processed / 'adult_icu_cohort_first24h.parquet'
cohort = pd.read_parquet(cohort_path)
assert len(cohort) == 45253 and cohort.ICUSTAY_ID.nunique() == 45253
assert not cohort[['ICUSTAY_ID','SUBJECT_ID','HADM_ID','INTIME']].isna().any().any()
cohort['INTIME'] = pd.to_datetime(cohort.INTIME).astype('datetime64[us]')
cohort['CUTOFF'] = cohort.INTIME + pd.Timedelta(hours=24)
cohort_index = cohort.set_index('ICUSTAY_ID', verify_integrity=True)
features = ['mechanical_ventilation_present', 'vasopressor_present', 'rrt_present']
print('Unchanged target cohort:', len(cohort), 'unique ICU stays')
print('Window: INTIME <= event time < INTIME + 24 hours; STORETIME < cutoff')


Unchanged target cohort: 45253 unique ICU stays
Window: INTIME <= event time < INTIME + 24 hours; STORETIME < cutoff


In [2]:
# Every code below is checked against the local MIMIC-III D_ITEMS dictionary.
VENT_SETTINGS = {445,448,449,450,1340,1486,1600,224687,
    639,654,681,682,683,684,224685,224684,224686,
    218,436,535,444,224697,224695,224696,224746,224747,
    221,1,1211,1655,2000,226873,224738,224419,224750,227187,
    543,5865,5866,224707,224709,224705,224706,
    60,437,505,506,686,220339,224700,3459,501,502,503,224702,
    223,667,668,669,670,671,672,224701}
VENT_SPECIAL = {720,223848,223849,467}
VENT_PROCEDURES = {225792}
VASO_CV = {'norepinephrine':{30047,30120}, 'epinephrine':{30044,30119,30309},
           'phenylephrine':{30127,30128}, 'vasopressin':{30051,42273,42802},
           'dopamine':{30043,30307}}
VASO_MV = {'norepinephrine':{221906}, 'epinephrine':{221289},
           'phenylephrine':{221749}, 'vasopressin':{222315}, 'dopamine':{221662}}
VASO_CV_IDS = set().union(*VASO_CV.values())
VASO_MV_IDS = set().union(*VASO_MV.values())
# Nonzero CRRT settings from crrt_durations (negative pressures are valid).
RRT_SETTINGS = {29,173,192,624,79,142,146,611,5683,
    224149,224144,228004,225183,225977,224154,224151,224150,
    225958,224145,224191,228005,228006,225976,224153,224152,226457}
RRT_POSITIVE = {226499,225810,225806,225807,225959,227639}
RRT_SPECIAL = {147,152,582,665,224146,225965,227290}
RRT_PROCEDURES = {225441,225802,225803,225805,225809,225955,225436}
# Dialysis-specific inputs from pivoted_rrt; exclude OR-only 44954.
RRT_INPUT_CV = {40788,40907,41063,41147,41307,41460,41620,41711,41791,41792,
    42562,43829,44037,44188,44526,44527,44584,44591,44698,44927,45157,45268,
    45352,45353,46012,46013,46172,46173,46250,46262,46292,46293,46311,
    46389,46574,46681,46720,46769,46773}
RRT_INPUT_MV = {227536,227525}
# Positive dialysis output from the repository's multi-table RRT concept.
RRT_OUTPUT = {40386,40425,40426,40507,40613,40624,40690,40745,40789,40881,
    40910,41016,41034,41069,41112,41250,41374,41417,41500,41527,41623,41635,
    41713,41750,41829,41842,42289,42388,42464,42524,42536,42868,42928,
    42972,43016,43052,43098,43115,43687,43941,44027,44085,44193,44199,
    44216,44286,44567,44843,44845,44857,44901,44943,45479,45828,46230,
    46232,46394,46464,46712,46713,46715,46741}
# OR-only output 41897 is excluded, consistently with the strict ICU window.
SOURCE_ITEMS = {
    'CHARTEVENTS': VENT_SETTINGS | VENT_SPECIAL | RRT_SETTINGS | RRT_POSITIVE | RRT_SPECIAL,
    'INPUTEVENTS_CV': VASO_CV_IDS | RRT_INPUT_CV,
    'INPUTEVENTS_MV': VASO_MV_IDS | RRT_INPUT_MV,
    'PROCEDUREEVENTS_MV': VENT_PROCEDURES | RRT_PROCEDURES,
    'OUTPUTEVENTS': RRT_OUTPUT,
}

def source_path(name):
    path = data_path / (name + '.csv')
    return path if path.is_file() else path / (name + '.csv')

dictionary = pd.read_csv(source_path('D_ITEMS'), usecols=['ITEMID','LABEL','LINKSTO'])
for table, ids in SOURCE_ITEMS.items():
    labels = dictionary.loc[dictionary.ITEMID.isin(ids)].copy()
    assert set(labels.ITEMID) == ids, (table, 'Unknown ITEMID')
    assert labels.LINKSTO.str.lower().eq(table.lower()).all(), (table, 'Wrong source table')
    print('\n' + table + ': ' + ', '.join(map(str, sorted(ids))))
    print(labels.sort_values('ITEMID').to_string(index=False))
print('\nVasopressor mapping (CareVue / MetaVision):')
for drug in VASO_CV:
    print(drug, sorted(VASO_CV[drug]), '/', sorted(VASO_MV[drug]))



CHARTEVENTS: 1, 29, 60, 79, 142, 146, 147, 152, 173, 192, 218, 221, 223, 436, 437, 444, 445, 448, 449, 450, 467, 501, 502, 503, 505, 506, 535, 543, 582, 611, 624, 639, 654, 665, 667, 668, 669, 670, 671, 672, 681, 682, 683, 684, 686, 720, 1211, 1340, 1486, 1600, 1655, 2000, 3459, 5683, 5865, 5866, 220339, 223848, 223849, 224144, 224145, 224146, 224149, 224150, 224151, 224152, 224153, 224154, 224191, 224419, 224684, 224685, 224686, 224687, 224695, 224696, 224697, 224700, 224701, 224702, 224705, 224706, 224707, 224709, 224738, 224746, 224747, 224750, 225183, 225806, 225807, 225810, 225958, 225959, 225965, 225976, 225977, 226457, 226499, 226873, 227187, 227290, 227639, 228004, 228005, 228006
 ITEMID                                             LABEL     LINKSTO
      1                                    % Inspir. Time chartevents
     29                                       Access mmHg chartevents
     60                                   Auto-PEEP Level chartevents
     79               

In [3]:
def numeric(series):
    # Empty CSV fields represent missing data; malformed nonempty values fail.
    return pd.to_numeric(series.replace('', pd.NA), errors='raise')

def valid_order(frame):
    cancel = numeric(frame.CANCELREASON).fillna(0)
    status = frame.STATUSDESCRIPTION.fillna('').str.lower()
    return cancel.eq(0) & ~status.isin(['rewritten','canceled','cancelled'])

def time_filter(frame, event_column):
    event_time = pd.to_datetime(frame[event_column].replace('', pd.NA), errors='raise')
    store_time = pd.to_datetime(frame.STORETIME.replace('', pd.NA), errors='raise')
    within = event_time.ge(frame.INTIME) & event_time.lt(frame.CUTOFF)
    available = store_time.notna() & store_time.lt(frame.CUTOFF)
    return within, available, event_time, store_time

def positive_mv_administration(frame):
    rate = numeric(frame.RATE)
    amount = numeric(frame.AMOUNT)
    end = pd.to_datetime(frame.ENDTIME.replace('', pd.NA), errors='raise')
    start = pd.to_datetime(frame.STARTTIME.replace('', pd.NA), errors='raise')
    # Do not use an amount accumulated beyond the 24-hour boundary.
    completed_amount = amount.gt(0) & end.ge(start) & end.lt(frame.CUTOFF)
    return rate.gt(0) | (rate.isna() & completed_amount)

def classify(table, frame):
    item = frame.ITEMID
    no = pd.Series(False, index=frame.index)
    vent, vaso, rrt = no.copy(), no.copy(), no.copy()
    if table == 'CHARTEVENTS':
        value = frame.VALUE.fillna('')
        number = numeric(frame.VALUENUM)
        valid = numeric(frame.ERROR).fillna(0).eq(0) & value.ne('')
        vent = valid & (item.isin(VENT_SETTINGS) |
            (item.eq(720) & value.ne('Other/Remarks')) |
            (item.eq(223848) & value.ne('Other')) | item.eq(223849) |
            (item.eq(467) & value.eq('Ventilator')))
        # Active treatment markers; access/catheter documentation alone is excluded.
        rrt = valid & ((item.isin(RRT_SETTINGS) & number.fillna(1).ne(0)) |
            (item.isin(RRT_POSITIVE) & number.gt(0)) |
            (item.eq(147) & value.eq('Yes')) |
            (item.isin([152,227290]) & value.isin(['CVVH','CVVHD','CVVHDF','SCUF','Peritoneal','Hemodialysis','CAVH','IHD'])) |
            (item.eq(582) & value.isin(['CAVH Start','CVVHD Start','Hemodialysis st','Peritoneal Dial'])) |
            (item.eq(665) & value.isin(['Initiated','Active','Clot Increasing','Clots Present','No Clot Present'])) |
            (item.eq(224146) & value.isin(['New Filter','Reinitiated'])) |
            (item.eq(225965) & value.eq('In use')))
    elif table == 'INPUTEVENTS_CV':
        rate, amount = numeric(frame.RATE), numeric(frame.AMOUNT)
        swapped = item.isin([42273,42802])
        effective_rate = rate.where(~swapped, amount)
        effective_amount = amount.where(~swapped, rate)
        vaso = item.isin(VASO_CV_IDS) & (effective_rate.gt(0) | (effective_rate.isna() & effective_amount.gt(0)))
        rrt = item.isin(RRT_INPUT_CV) & amount.gt(0)
    elif table == 'INPUTEVENTS_MV':
        administered = valid_order(frame) & positive_mv_administration(frame)
        vaso = item.isin(VASO_MV_IDS) & administered
        rrt = item.isin(RRT_INPUT_MV) & administered
    elif table == 'PROCEDUREEVENTS_MV':
        valid = valid_order(frame)
        vent = item.isin(VENT_PROCEDURES) & valid
        rrt = item.isin(RRT_PROCEDURES) & valid
    elif table == 'OUTPUTEVENTS':
        rrt = item.isin(RRT_OUTPUT) & numeric(frame.VALUE).gt(0) & numeric(frame.ISERROR).fillna(0).eq(0)
    return dict(zip(features, [vent.fillna(False),vaso.fillna(False),rrt.fillna(False)]))

# Boundary, availability, and treatment-definition regression checks.
t = pd.Timestamp('2100-01-01')
sample = pd.DataFrame({'INTIME':[t]*5, 'CUTOFF':[t+pd.Timedelta(days=1)]*5,
    'CHARTTIME':[t-pd.Timedelta(seconds=1), t, t+pd.Timedelta(days=1), t, t],
    'STORETIME':[t,t,t,t+pd.Timedelta(days=1),pd.NaT]})
a,b,_,_ = time_filter(sample, 'CHARTTIME')
assert (a & b).tolist() == [False,True,False,False,False]
cv_sample = pd.DataFrame({'ITEMID':[30047,30047,42273,30047,30047],
    'RATE':['1','0','','','-1'], 'AMOUNT':['','5','0.04','2','3']})
assert classify('INPUTEVENTS_CV',cv_sample)['vasopressor_present'].tolist() == [True,False,True,True,False]
chart_sample = pd.DataFrame({'ITEMID':[467,467,147,147,224154,224154,720,720,223848],
    'VALUE':['Cannula','Ventilator','No','Yes','100','0','Other/Remarks','Assist Control','Other'],
    'VALUENUM':['','','','','100','0','','',''], 'ERROR':['0']*9})
assert classify('CHARTEVENTS',chart_sample)['mechanical_ventilation_present'].tolist() == [False,True,False,False,False,False,False,True,False]
assert classify('CHARTEVENTS',chart_sample)['rrt_present'].tolist() == [False,False,False,True,True,False,False,False,False]
mv_sample = pd.DataFrame({'ITEMID':[221906]*4, 'RATE':['1','1','',''], 'AMOUNT':['','', '2','2'],
    'STARTTIME':[t]*4, 'ENDTIME':[t+pd.Timedelta(hours=30), t+pd.Timedelta(hours=1),t+pd.Timedelta(hours=30),t+pd.Timedelta(hours=1)],
    'CUTOFF':[t+pd.Timedelta(days=1)]*4, 'CANCELREASON':['0','1','0','0'], 'STATUSDESCRIPTION':['FinishedRunning']*4})
assert classify('INPUTEVENTS_MV',mv_sample)['vasopressor_present'].tolist() == [True,False,False,True]
print('Passed: time boundaries, late/missing entry exclusion, positive dose, special vasopressin fields, canceled orders, future amount exclusion and negative treatment examples.')


Passed: time boundaries, late/missing entry exclusion, positive dose, special vasopressin fields, canceled orders, future amount exclusion and negative treatment examples.


In [4]:
# Bounded-memory scan: project source IDs before converting selected batches to pandas.
# No existing feature file is opened for writing.
COLUMNS = {
    'CHARTEVENTS':['CHARTTIME','STORETIME','VALUE','VALUENUM','ERROR'],
    'INPUTEVENTS_CV':['CHARTTIME','STORETIME','RATE','AMOUNT'],
    'INPUTEVENTS_MV':['STARTTIME','STORETIME','ENDTIME','RATE','AMOUNT','CANCELREASON','STATUSDESCRIPTION'],
    'PROCEDUREEVENTS_MV':['STARTTIME','STORETIME','CANCELREASON','STATUSDESCRIPTION'],
    'OUTPUTEVENTS':['CHARTTIME','STORETIME','VALUE','ISERROR'],
}
positive_stays = {feature:set() for feature in features}
witnesses = {feature:{} for feature in features}
source_audits = []
item_evidence_counts = Counter()
rrt_mode_values = Counter()
source_signatures = {}
started = time.monotonic()
for table in ['PROCEDUREEVENTS_MV','INPUTEVENTS_MV','INPUTEVENTS_CV','OUTPUTEVENTS','CHARTEVENTS']:
    raw = source_path(table)
    stat = raw.stat()
    source_signatures[table] = (stat.st_size, stat.st_mtime_ns)
    columns = ['ICUSTAY_ID','SUBJECT_ID','HADM_ID','ITEMID'] + COLUMNS[table]
    allowed_items = pa.array([str(i) for i in SOURCE_ITEMS[table]])
    allowed_stays = pa.array([str(i) for i in cohort.ICUSTAY_ID])
    event_column = 'STARTTIME' if 'STARTTIME' in columns else 'CHARTTIME'
    audit = dict(table=table, scanned=0, selected_cohort_rows=0, mismatched_identifiers=0, missing_auxiliary_identifiers=0, outside_or_missing_event_time=0,
                 late_or_missing_storetime=0, qualifying_time_rows=0)
    source_positive = {feature:set() for feature in features}
    next_progress = time.monotonic() + 45
    print('Scanning', table, flush=True)
    with raw.open('rb') as handle:
        with pacsv.open_csv(handle, read_options=pacsv.ReadOptions(block_size=1024*1024, use_threads=False),
             parse_options=pacsv.ParseOptions(newlines_in_values=True),
             convert_options=pacsv.ConvertOptions(include_columns=columns, column_types={c:pa.string() for c in columns})) as reader:
            for batch in reader:
                audit['scanned'] += batch.num_rows
                selected = batch.filter(pc.and_(pc.is_in(batch.column('ITEMID'), value_set=allowed_items),
                                                pc.is_in(batch.column('ICUSTAY_ID'), value_set=allowed_stays)))
                if selected.num_rows:
                    frame = selected.to_pandas()
                    for c in ['ICUSTAY_ID','SUBJECT_ID','HADM_ID','ITEMID']:
                        frame[c] = numeric(frame[c]).astype('Int64')
                    # Map by ICU stay, then verify the associated patient and admission.
                    audit['missing_auxiliary_identifiers'] += int(frame[['SUBJECT_ID','HADM_ID']].isna().any(axis=1).sum())
                    matching_ids = (frame.SUBJECT_ID.isna() | frame.SUBJECT_ID.eq(frame.ICUSTAY_ID.map(cohort_index.SUBJECT_ID))) & (frame.HADM_ID.isna() | frame.HADM_ID.eq(frame.ICUSTAY_ID.map(cohort_index.HADM_ID)))
                    matching_ids = matching_ids.fillna(False)
                    audit['mismatched_identifiers'] += int((~matching_ids).sum())
                    frame = frame.loc[matching_ids].copy()
                    frame['INTIME'] = frame.ICUSTAY_ID.map(cohort_index.INTIME)
                    frame['CUTOFF'] = frame.ICUSTAY_ID.map(cohort_index.CUTOFF)
                    within, available, event_time, store_time = time_filter(frame,event_column)
                    audit['selected_cohort_rows'] += len(frame)
                    audit['outside_or_missing_event_time'] += int((~within).sum())
                    audit['late_or_missing_storetime'] += int((within & ~available).sum())
                    frame['EVENT_TIME'] = event_time
                    frame['RECORDED_TIME'] = store_time
                    frame = frame.loc[within & available].copy()
                    audit['qualifying_time_rows'] += len(frame)
                    if len(frame):
                        if table == 'CHARTEVENTS':
                            for (item,value),n in frame.loc[frame.ITEMID.isin([152,227290])].groupby(['ITEMID','VALUE']).size().items():
                                rrt_mode_values[(int(item),value)] += int(n)
                        for feature, mask in classify(table,frame).items():
                            evidence = frame.loc[mask]
                            ids = set(evidence.ICUSTAY_ID)
                            source_positive[feature].update(ids)
                            new = ids - positive_stays[feature]
                            if new:
                                first = evidence.loc[evidence.ICUSTAY_ID.isin(new)].drop_duplicates('ICUSTAY_ID')
                                for row in first.itertuples():
                                    witnesses[feature][row.ICUSTAY_ID] = (table,int(row.ITEMID),row.EVENT_TIME,row.RECORDED_TIME)
                            positive_stays[feature].update(ids)
                            for item,n in evidence.groupby('ITEMID').size().items():
                                item_evidence_counts[(table,feature,int(item))] += int(n)
                if time.monotonic() >= next_progress:
                    print(table, 'rows:', f"{audit['scanned']:,}", 'positive stays:', {f:len(v) for f,v in positive_stays.items()}, flush=True)
                    next_progress = time.monotonic() + 45
    assert source_signatures[table] == (raw.stat().st_size, raw.stat().st_mtime_ns), 'Source changed during scan'
    audit.update({f:len(v) for f,v in source_positive.items()})
    source_audits.append(audit)
    print('Completed',table, audit, flush=True)
print('Scan elapsed seconds:', round(time.monotonic()-started,1))


Scanning PROCEDUREEVENTS_MV
Completed PROCEDUREEVENTS_MV {'table': 'PROCEDUREEVENTS_MV', 'scanned': 258066, 'selected_cohort_rows': 14487, 'mismatched_identifiers': 0, 'missing_auxiliary_identifiers': 0, 'outside_or_missing_event_time': 6064, 'late_or_missing_storetime': 4707, 'qualifying_time_rows': 3716, 'mechanical_ventilation_present': 3213, 'vasopressor_present': 0, 'rrt_present': 276}
Scanning INPUTEVENTS_MV
Completed INPUTEVENTS_MV {'table': 'INPUTEVENTS_MV', 'scanned': 3618991, 'selected_cohort_rows': 244692, 'mismatched_identifiers': 0, 'missing_auxiliary_identifiers': 0, 'outside_or_missing_event_time': 177349, 'late_or_missing_storetime': 2823, 'qualifying_time_rows': 64520, 'mechanical_ventilation_present': 0, 'vasopressor_present': 5634, 'rrt_present': 123}
Scanning INPUTEVENTS_CV
INPUTEVENTS_CV rows: 15,576,838 positive stays: {'mechanical_ventilation_present': 3213, 'vasopressor_present': 13384, 'rrt_present': 368}
Completed INPUTEVENTS_CV {'table': 'INPUTEVENTS_CV', 'sc

In [5]:
# Final table preserves exactly the existing cohort IDs and row order.
intervention_features = cohort[['ICUSTAY_ID']].copy()
for feature in features:
    intervention_features[feature] = intervention_features.ICUSTAY_ID.isin(positive_stays[feature]).astype('int8')
assert intervention_features.columns.tolist() == ['ICUSTAY_ID'] + features
assert len(intervention_features) == intervention_features.ICUSTAY_ID.nunique() == 45253
assert intervention_features.ICUSTAY_ID.equals(cohort.ICUSTAY_ID)
assert not intervention_features.isna().any().any()
assert intervention_features[features].isin([0,1]).all().all()

# Independently recheck one actual supporting event for EVERY positive flag.
for feature in features:
    assert set(witnesses[feature]) == positive_stays[feature]
    for stay,(table,item,event_time,store_time) in witnesses[feature].items():
        intime = cohort_index.at[stay,'INTIME']
        cutoff = intime + pd.Timedelta(hours=24)
        assert intime <= event_time < cutoff
        assert pd.notna(store_time) and store_time < cutoff
        assert item in SOURCE_ITEMS[table]
    assert int(intervention_features[feature].sum()) == len(witnesses[feature])
assert {str(p):digest(p) for p in protected_paths} == protected_before, 'An existing artifact changed'
partial_path = output_path.with_suffix('.parquet.partial')
intervention_features.to_parquet(partial_path,index=False)
pd.testing.assert_frame_equal(pd.read_parquet(partial_path), intervention_features)
partial_path.replace(output_path)
assert {str(p):digest(p) for p in protected_paths} == protected_before
summary = pd.DataFrame({'feature':features,
    'present_1':[int(intervention_features[f].sum()) for f in features],
    'absent_0':[int((intervention_features[f]==0).sum()) for f in features]})
summary['percent_present'] = (summary.present_1 / len(cohort) * 100).round(2)
print(summary.to_string(index=False))
print('\nSaved:', output_path)
print('Shape:', intervention_features.shape, '| Duplicate ICU IDs: 0 | Missing values: 0')
print('Every positive feature has an audited first-24-hour, pre-cutoff stored witness.')
print('SHA-256 checks passed: all existing cohort, lab, vital, output and other notebook files unchanged.')
print('\nSource audit:')
print(pd.DataFrame(source_audits).to_string(index=False))
print('\nObserved in-window dialysis mode values:')
print(dict(rrt_mode_values))
print('\nPositive evidence counts per ITEMID (documentation only, not additional features):')
for key,n in sorted(item_evidence_counts.items()):
    print(key,n)


                       feature  present_1  absent_0  percent_present
mechanical_ventilation_present      21660     23593            47.86
           vasopressor_present      13666     31587            30.20
                   rrt_present       1161     44092             2.57

Saved: /home/sengul/projects/icu-risk-prediction/data/processed/intervention_features_first24h.parquet
Shape: (45253, 4) | Duplicate ICU IDs: 0 | Missing values: 0
Every positive feature has an audited first-24-hour, pre-cutoff stored witness.
SHA-256 checks passed: all existing cohort, lab, vital, output and other notebook files unchanged.

Source audit:
             table   scanned  selected_cohort_rows  mismatched_identifiers  missing_auxiliary_identifiers  outside_or_missing_event_time  late_or_missing_storetime  qualifying_time_rows  mechanical_ventilation_present  vasopressor_present  rrt_present
PROCEDUREEVENTS_MV    258066                 14487                       0                              0        

## Interpretation and limitations

- Exactly three binary features are written. Per-table counts, source code lists and timestamp audits above are notebook diagnostics only.
- No later chart entries, late STORETIME entries, discharge diagnoses or treatment duration beyond 24 hours are used. A treatment begun before ICU entry is not carried forward without an in-window qualifying record. Missing STORETIME is conservatively excluded.
- Ventilation settings are the standard repository proxy; they do not perfectly distinguish invasive ventilation from all possible NIV documentation. The procedure supplement is explicitly invasive ventilation.
- RRT-specific fluids and CRRT medications are indirect evidence of therapy, consistent with the repository. Catheter-only documentation, end-only CareVue procedure labels and OR-only codes are deliberately excluded. This improves specificity but may lower sensitivity relative to the broad first-day RRT SQL.
- Vasopressor presence includes any positive administration, including documented boluses; it is not a shock or sustained-infusion diagnosis. Low-dose dopamine is included; dobutamine and milrinone are not.
- The original cohort and all existing lab, vital and output files/notebooks were verified unchanged by SHA-256. No Git commit or push is performed.

### Attribution

ITEMID groupings and value predicates are adapted from the MIT-LCP MIMIC Code Repository (MIT License):

Copyright (c) 2019 MIT Laboratory for Computational Physiology

Permission is hereby granted, free of charge, to any person obtaining a copy of this software and associated documentation files (the "Software"), to deal in the Software without restriction, including without limitation the rights to use, copy, modify, merge, publish, distribute, sublicense, and/or sell copies of the Software, and to permit persons to whom the Software is furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY, FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM, OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE SOFTWARE.


In [2]:
intervention_features_df = pd.read_parquet(
    "../data/processed/intervention_features_first24h.parquet"
)

intervention_features_df.head()

,ICUSTAY_ID,mechanical_ventilation_present,vasopressor_present,rrt_present
0,280836,1,0,0
1,206613,0,1,0
2,220345,0,1,0
3,249196,0,0,0
4,210407,0,0,0


In [3]:
intervention_features_df.shape

(45253, 4)

In [4]:
intervention_features_df["ICUSTAY_ID"].nunique()

45253

In [5]:
intervention_features_df.isna().sum()

ICUSTAY_ID                        0
mechanical_ventilation_present    0
vasopressor_present               0
rrt_present                       0
dtype: int64